# Exploratory Data Analysis (EDA)

## Definition
**EDA** is the process of **analyzing and summarizing** a dataset to understand its characteristics
before modeling. It reveals:
- Structure and schema
- Missing data patterns
- Feature distributions
- Relationships between variables
- Outliers and anomalies
- Target variable characteristics

### EDA Workflow
1. Load & inspect basic info
2. Analyze each feature (univariate)
3. Analyze feature-target relationships (bivariate)
4. Analyze feature-feature relationships (multivariate)
5. Summarize findings and plan preprocessing

In [ ]:
import pandas as pd
import numpy as np

# Load a synthetic dataset
rng = np.random.default_rng(42)
n = 500

df = pd.DataFrame({
    'age':         rng.integers(18, 70, n),
    'income':      rng.lognormal(11, 0.5, n).clip(0, 500000),
    'credit_score': rng.normal(700, 80, n).clip(300, 850).astype(int),
    'loan_amount': rng.lognormal(10, 0.6, n).clip(1000, 200000),
    'employment':  rng.choice(['Employed', 'Self-Employed', 'Unemployed', 'Retired'], n,
                               p=[0.6, 0.2, 0.1, 0.1]),
    'education':   rng.choice(['High School', 'Bachelor', 'Master', 'PhD'], n,
                               p=[0.3, 0.4, 0.2, 0.1]),
    'default':     rng.choice([0, 1], n, p=[0.85, 0.15])
})
# Inject some missing values
df.loc[rng.choice(n, 30, replace=False), 'income'] = np.nan
df.loc[rng.choice(n, 15, replace=False), 'credit_score'] = np.nan
print('Dataset loaded:', df.shape)

In [ ]:
# Step 1: Basic inspection
print('=== BASIC INFO ===')
print(f'Shape: {df.shape}')
print(f'Duplicates: {df.duplicated().sum()}')
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:')
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(missing_df[missing_df['count'] > 0])

In [ ]:
# Step 2: Statistical summary
print('=== DESCRIPTIVE STATISTICS ===')
print(df.describe().round(2))
print()
print('Categorical value counts:')
for col in df.select_dtypes(include='object').columns:
    print(f'\n{col}:')
    print(df[col].value_counts())

In [ ]:
# Step 3: Target variable analysis
print('=== TARGET VARIABLE ===')
print('Default distribution:')
print(df['default'].value_counts())
print('Default rate:', df['default'].mean().round(3))

# Default rate by categorical feature
print('\nDefault rate by employment:')
print(df.groupby('employment')['default'].agg(['mean', 'count']).round(3))

In [ ]:
# Step 4: Numeric feature analysis
print('=== NUMERIC FEATURES ===')
num_cols = df.select_dtypes(include='number').columns.drop('default')
for col in num_cols:
    q25 = df[col].quantile(0.25)
    q75 = df[col].quantile(0.75)
    print(f'{col:20s}: mean={df[col].mean():.0f}  std={df[col].std():.0f}  '
          f'skew={df[col].skew():.2f}  IQR=[{q25:.0f},{q75:.0f}]')

In [ ]:
# Step 5: Feature-target relationship
print('=== FEATURE-TARGET ANALYSIS ===')
num_cols_no_target = ['age', 'income', 'credit_score', 'loan_amount']
for col in num_cols_no_target:
    defaulted = df[df['default']==1][col].mean()
    not_defaulted = df[df['default']==0][col].mean()
    print(f'{col:20s}: Default mean={defaulted:.0f}  Non-default mean={not_defaulted:.0f}')

In [ ]:
# Step 6: Correlation analysis
print('=== CORRELATIONS ===')
corr = df[num_cols_no_target + ['default']].corr()['default'].drop('default').sort_values()
print(corr.round(3))

In [ ]:
# Step 7: EDA summary function
def quick_eda(df, target_col):
    print(f'Shape: {df.shape}')
    print(f'Missing: {df.isna().sum().sum()} values in {df.isna().any().sum()} cols')
    print(f'Duplicates: {df.duplicated().sum()}')
    if target_col in df.columns:
        print(f'Target balance: {df[target_col].value_counts(normalize=True).round(3).to_dict()}')
    num_cols = df.select_dtypes(include='number').columns.drop(target_col, errors='ignore')
    corr = df[num_cols].corrwith(df[target_col]).abs().sort_values(ascending=False)
    print(f'Top correlated features:\n{corr.head(5).round(3)}')

quick_eda(df, 'default')